In [1]:
!mkdir -p names

In [2]:
import os
import random
import string
import torch
import torch.nn as nn

In [3]:
ALL_LETTERS = string.ascii_letters + " .,;'-"
EOS = '\n'
VOCAB = ALL_LETTERS + EOS
N_CHARS = len(VOCAB)

char_to_idx = {c: i for i, c in enumerate(VOCAB)}
idx_to_char = {i: c for i, c in enumerate(VOCAB)}

print(f"Vocab size: {N_CHARS}")
print(f"Example: 'A' -> {char_to_idx['A']}, EOS -> {char_to_idx[EOS]}")

Vocab size: 59
Example: 'A' -> 26, EOS -> 58


In [4]:
data_dir = "names"

names = []
for filename in os.listdir(data_dir):
    if not filename.endswith(".txt"):
        continue
    with open(os.path.join(data_dir, filename), encoding='utf-8') as f:
        for line in f:
            name = line.strip()
            if name and all(c in ALL_LETTERS for c in name):
                names.append(name)

print(f"Total names loaded: {len(names)}")
print(f"Examples: {random.sample(names, min(5, len(names)))}")

Total names loaded: 19912
Examples: ['Nikolaou', 'Dubnov', 'Tomonaga', 'Turtsevich', 'Khouri']


In [5]:
def name_to_tensors(name):
    chars = list(name) + [EOS]
    indices = [char_to_idx[c] for c in chars]
    input_tensor = torch.tensor(indices[:-1], dtype=torch.long)
    target_tensor = torch.tensor(indices[1:], dtype=torch.long)
    return input_tensor, target_tensor

# Test
x, y = name_to_tensors("Ali")
print(f"Input : {x} → chars: {[idx_to_char[i.item()] for i in x]}")
print(f"Target: {y} → chars: {[idx_to_char[i.item()] for i in y]}")

Input : tensor([26, 11,  8]) → chars: ['A', 'l', 'i']
Target: tensor([11,  8, 58]) → chars: ['l', 'i', '\n']


In [6]:
class NameRNN(nn.Module):
    def __init__(self, vocab_size, hidden_size):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, hidden_size) # (batch, seq_len, hidden_size)
        self.rnn = nn.RNN(hidden_size, hidden_size, batch_first=True)
        self.out = nn.Linear(hidden_size, vocab_size)

    def forward(self, x, hidden=None):
        """
        x      : (batch, seq_len)  — token indices
        hidden : (1, batch, hidden_size) — optional previous hidden state

        Returns:
            logits : (batch, seq_len, vocab_size)
            hidden : (1, batch, hidden_size)
        """
        x = self.emb(x)              # (batch, seq_len, hidden_size)
        out, hidden = self.rnn(x, hidden)  # out: (batch, seq_len, hidden_size)
        logits = self.out(out)       # (batch, seq_len, vocab_size)
        return logits, hidden

In [7]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

model = NameRNN(N_CHARS, hidden_size=128).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.005)
criterion = nn.CrossEntropyLoss()

n_epochs = 20
batch_size = 32

Using device: cuda


In [8]:
for epoch in range(1, n_epochs + 1):
    model.train()
    indices = list(range(len(names)))
    random.shuffle(indices)
    batches = [indices[i:i + batch_size] for i in range(0, len(indices), batch_size)]

    epoch_loss = 0.0
    for batch in batches:
        batch_loss = 0.0
        for i in batch:
            x, y = name_to_tensors(names[i])
            x = x.unsqueeze(0).to(device)  # (1, seq_len)
            y = y.to(device)               # (seq_len,)

            logits, _ = model(x)           # (1, seq_len, vocab_size)
            loss = criterion(logits.squeeze(0), y)
            batch_loss += loss

        optimizer.zero_grad()
        batch_loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=3)
        optimizer.step()

        epoch_loss += batch_loss.item() / len(batch)

    avg_loss = epoch_loss / len(batches)
    print(f"Epoch {epoch:>2}/{n_epochs}  |  avg loss = {avg_loss:.4f}")

Epoch  1/20  |  avg loss = 2.2134
Epoch  2/20  |  avg loss = 2.0691
Epoch  3/20  |  avg loss = 2.0272
Epoch  4/20  |  avg loss = 2.0021
Epoch  5/20  |  avg loss = 1.9821
Epoch  6/20  |  avg loss = 1.9707
Epoch  7/20  |  avg loss = 1.9589
Epoch  8/20  |  avg loss = 1.9561
Epoch  9/20  |  avg loss = 1.9478
Epoch 10/20  |  avg loss = 1.9422
Epoch 11/20  |  avg loss = 1.9409
Epoch 12/20  |  avg loss = 1.9398
Epoch 13/20  |  avg loss = 1.9365
Epoch 14/20  |  avg loss = 1.9373
Epoch 15/20  |  avg loss = 1.9364
Epoch 16/20  |  avg loss = 1.9393
Epoch 17/20  |  avg loss = 1.9369
Epoch 18/20  |  avg loss = 1.9381
Epoch 19/20  |  avg loss = 1.9389
Epoch 20/20  |  avg loss = 1.9403


In [9]:
@torch.no_grad()
def generate(start_letter='A', max_length=20, temperature=1.0):
    model.eval()

    idx = torch.tensor([[char_to_idx[start_letter]]], device=device)  # (1, 1)
    hidden = None
    result = start_letter

    for _ in range(max_length):
        logits, hidden = model(idx, hidden)  # logits: (1, 1, vocab_size)
        logits = logits[0, -1] / temperature  # (vocab_size,)
        probs = torch.softmax(logits, dim=-1)
        next_idx = torch.multinomial(probs, num_samples=1).item()
        next_char = idx_to_char[next_idx]

        if next_char == EOS:
            break

        result += next_char
        idx = torch.tensor([[next_idx]], device=device)

    return result

In [10]:
letters = list("ABCDEFGHIJKLMNOPQRS")

print("=" * 40)
print("         Generated Names")
print("=" * 40)

for letter in letters:
    generated = [generate(letter, temperature=0.8) for _ in range(3)]
    print(f"  {letter}: {', '.join(generated)}")

print("=" * 40)

         Generated Names
  A: Ausogron, Andruhin, Adak
  B: Babai, Baruskov, Babu
  C: Cham, Charvan, Chun
  D: Dabin, Derrik, Dubilov
  E: Emzham, Eiher, Elum
  F: Falehant, Fardan, Fordoenkov
  G: Gafertys, Gattak, Getley
  H: Harazzhuk, Handaginin, Han
  I: Isepenko, Ijin, Itin
  J: Jachov, Jakunin, Jittin
  K: Kurashi, Kowlo, Kayamov
  L: Ley, Lammondridgent, Lamsinsky
  M: Mui, Millor, Moranin
  N: Nina, Nassar, Natimel
  O: Obi, Odera, Oba
  P: Porton, Pascos, Pazov
  Q: Quradin, Qurubenin, Quak
  R: Rivitov, Ryum, Rundzin
  S: Shamon, Said, Sanimanov


In [27]:
# Temperature ta'sirini ko'rish
start = 'S'
print(f"\nTemperature comparison for letter '{start}':")
print("-" * 40)
for temp in [0.1, 0.4, 0.8, 1.2, 1.6, 2, 3, 10]:
    examples = [generate(start, temperature=temp) for _ in range(5)]
    print(f"  temp={temp}: {', '.join(examples)}")


Temperature comparison for letter 'S':
----------------------------------------
  temp=0.1: Sarraf, Sabbag, Sabbagh, Sarraf, Sarraf
  temp=0.4: Shamanev, Sarraf, Shirov, Shamanov, Sarkov
  temp=0.8: Shadensky, Shaian, Sarraf, Shamandy, Staidarr
  temp=1.2: Sladgaev, Saitama, Sdimurin, Stebi, Shamakyan
  temp=1.6: Shanefouk, Saidud, Staraevsky, Satmadd, Shamidikhlsnty
  temp=2: Sonfayuk, Svivaozoof ,Ko.boud, Stednichinsdolkiro, Stumondzlasc, Sh'chmantedersy
  temp=3: Sgruj, ShBewnyjiolbersayarah, Suimggjksinbmctur, Shvo, SqRolsansaff
  temp=10: Sstko, S,psjx, Szlc .leid, S,ckyeando-StpucfujHb, ShHtLspfa NKtucotyd


In [12]:
# Save
torch.save(model.state_dict(), "rnn_name_model.pt")
print("Model saved to rnn_name_model.pt")

# Load
loaded_model = NameRNN(N_CHARS, hidden_size=128).to(device)
loaded_model.load_state_dict(torch.load("rnn_name_model.pt", map_location=device))
loaded_model.eval()
print("Model loaded successfully!")

# Test loaded model
print("\nTest generation from loaded model:")
print([generate('A', temperature=0.8) for _ in range(5)])

Model saved to rnn_name_model.pt
Model loaded successfully!

Test generation from loaded model:
['Almons', 'Adramonov', 'Asghar', 'Ani', 'Abala']
